# 第 2 天 · Python 关键语法（用 Java 对照学）

> 📓 **这是第 2 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/2)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
print("✓ 环境就绪，可以往下运行了")

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Order:
    order_id: str
    amount: float
    status: str = "NEW"
    tags: list[str] = field(default_factory=list)

    @property
    def is_big(self) -> bool:     # ≈ getter，调用时不加括号：o.is_big
        return self.amount >= 1000


@dataclass(frozen=True)
class Money:
    value: int
    currency: str = "CNY"


o = Order("A002", 1200.0, tags=["vip"])
print(o)                           # 自动 toString
print(o.is_big)
print(Money(100) == Money(100))    # 自动 equals → True

m = Money(1)
try:
    m.value = 2                    # frozen：不可修改
except Exception as e:
    print("不可变：", type(e).__name__)

In [ ]:
nums = [3, 8, 1, 12, 7, 20]
evens_squared = [n * n for n in nums if n % 2 == 0]
index = {n: str(n) * 2 for n in nums}
print(evens_squared)
print(index)
print(sum(n for n in nums if n > 5))   # 生成器表达式：不建中间列表，≈ 惰性 Stream

In [ ]:
from contextlib import contextmanager


@contextmanager
def transaction(name: str):
    print(f"[tx] BEGIN {name}")
    try:
        yield                      # with 块里的代码在这里执行
        print(f"[tx] COMMIT {name}")
    except Exception:
        print(f"[tx] ROLLBACK {name}")
        raise


with transaction("ok"):
    print("  做点事")

try:
    with transaction("boom"):
        raise ValueError("出错了")
except ValueError as e:
    print("外层捕获：", e)

In [ ]:
import functools
import time


def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):           # *args ≈ 可变参数，**kwargs ≈ 命名参数 Map
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[timed] {func.__name__} 耗时 {(time.perf_counter() - start) * 1000:.2f} ms")
    return wrapper


def retry(times: int = 3):                  # 第一层：接收装饰器参数
    def decorator(func):                    # 第二层：接收被装饰的函数
        @functools.wraps(func)
        def wrapper(*args, **kwargs):       # 第三层：真正执行
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ValueError as e:
                    print(f"[retry] 第 {attempt} 次失败：{e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator


calls = 0


@timed
@retry(times=3)
def flaky() -> str:
    global calls
    calls += 1
    if calls < 3:
        raise ValueError("网关超时")
    return "成功"


print(flaky())

In [ ]:
"""第 2 天：Python 关键语法（Java 对照版）。运行：uv run day02_basics.py"""
from __future__ import annotations

import functools
import time
from contextlib import contextmanager
from dataclasses import dataclass, field


# 1. 类型注解 + dataclass  ≈ Java record / Lombok @Data
@dataclass
class Order:
    order_id: str
    amount: float
    status: str = "NEW"                                # 默认值
    tags: list[str] = field(default_factory=list)      # 可变默认值必须这样写

    @property
    def is_big(self) -> bool:                          # ≈ getter
        return self.amount >= 1000


@dataclass(frozen=True)                                # ≈ 不可变 record
class Money:
    value: int
    currency: str = "CNY"


# 2. 自定义异常  ≈ class OrderException extends RuntimeException
class OrderError(Exception):
    def __init__(self, order_id: str, reason: str) -> None:
        super().__init__(f"订单 {order_id} 异常：{reason}")
        self.order_id = order_id


# 3. 装饰器  ≈ Spring AOP 的 @Around
def timed(func):
    @functools.wraps(func)                             # 保留原函数名和文档
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            cost = (time.perf_counter() - start) * 1000
            print(f"[timed] {func.__name__} 耗时 {cost:.2f} ms")
    return wrapper


def retry(times: int = 3):                             # 带参数的装饰器 = 再包一层
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except OrderError as e:
                    print(f"[retry] 第 {attempt} 次失败：{e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator


# 4. 上下文管理器  ≈ try-with-resources + AutoCloseable
@contextmanager
def transaction(name: str):
    print(f"[tx] BEGIN {name}")
    try:
        yield                                          # with 块里的代码在这里执行
        print(f"[tx] COMMIT {name}")
    except Exception:
        print(f"[tx] ROLLBACK {name}")
        raise


_calls = {"n": 0}


@retry(times=3)
def pay(order: Order) -> str:
    _calls["n"] += 1
    if _calls["n"] < 3:                                # 前两次故意失败
        raise OrderError(order.order_id, "支付网关超时")
    order.status = "PAID"
    return f"{order.order_id} 支付成功"


@timed
def main() -> None:
    orders = [
        Order("A001", 99.5),
        Order("A002", 1200.0, tags=["vip"]),
        Order("A003", 3500.0),
        Order("A004", 15.0, status="CANCELLED"),
    ]
    print(orders[1])                                   # dataclass 自带 toString

    # 5. 推导式  ≈ Stream API
    big_ids = [o.order_id for o in orders if o.is_big]                   # filter + map + toList
    amount_by_id = {o.order_id: o.amount for o in orders}                # Collectors.toMap
    total = sum(o.amount for o in orders if o.status != "CANCELLED")     # mapToDouble().sum()
    print("大额订单：", big_ids)
    print("金额表：", amount_by_id)
    print(f"有效总额：{total:.2f}")

    print(Money(100) == Money(100))                    # True，自动 equals

    # 6. 异常处理 + with
    with transaction("pay-A002"):
        print(pay(orders[1]))

    try:
        with transaction("refund-A004"):
            raise OrderError("A004", "已取消订单不能退款")
    except OrderError as e:                            # ≈ catch (OrderError e)
        print("捕获到：", e, "| order_id =", e.order_id)
    else:
        print("没有异常时才执行")
    finally:
        print("finally 一定执行")


if __name__ == "__main__":                             # ≈ public static void main
    main()

In [ ]:
from dataclasses import dataclass
import functools


def log_args(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # TODO: 打印入参，调用 func，打印返回值
        return func(*args, **kwargs)
    return wrapper


@dataclass
class LRUCache:
    capacity: int
    # TODO: 用 dict 实现（Python 3.7+ 的 dict 保持插入顺序）


@log_args
def demo() -> None:
    print("开始写你的工具类吧")


demo()

In [ ]:
from dataclasses import dataclass

# 经典坑：函数默认值只在定义时创建一次，所有调用共享同一个列表
def f(x=[]):
    x.append(1)
    return x

print(f(), f(), f())

# dataclass 直接拦住了这种写法
try:
    @dataclass
    class Bad:
        tags: list[str] = []
except ValueError as e:
    print("dataclass 报错：", e)